# Do the estimates move when institution type is included?

In the shortlist notebook, each institution's **estimated true value added** in a subject was plotted against its **published** value added, and the correction (the distance from the diagonal) showed where small cohorts had been pulled toward what the rest of the evidence supports. That model knew whether an institution was independent, a college or something else, but treated all state schools alike. This notebook asks whether **the estimated axis moves** once the state types are included: academy converter, academy sponsor-led, free school / UTC / studio and LA maintained.

**Two fits, everything else the same.** The England-wide model on all 2,747 institutions and the seven high-level subject groups (Maths, Sciences, English, Humanities, Social sciences, Business & Computing, Creative arts), with region and local-authority effects, the Maths-and-Science versus English-and-Open tilt, and separate scatter for institutions without GCSE results:

- **Without state types:** no differences among the state types. Independent schools, colleges and other institutions keep their own offsets by subject group, as in the shortlist notebook.
- **With state types:** shifts by state type in GCSE general quality, tilt and consistency, in A-level value added beyond GCSE for each subject group, and a trend with time as an academy (as in `a-level-institution-type.ipynb`).

From each fit we compute every institution's estimated true value added in each group, with an interval, and compare the two. **What to look for:** points that leave the diagonal in the plots of "without type" against "with type", which institutions and types move, and whether the top of each list changes.

**A caution on reading movement.** An institution's estimate moves when the model learns that its type differs from the average state school: a sponsor-led academy is pulled down, a converter up, and the pull is strongest for small cohorts, which lean on the model's expectation. A shift is not a verdict on the institution; it says how much of its published score the model attributes to the type of school it is, rather than to that school specifically.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import xarray as xr

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
print(f"Running on PyMC v{pm.__version__}")

## Data

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")
names = pd.read_csv("data/school-names.csv").set_index("URN")

# ---- GCSE: six elements per school with known standard errors (schools with Progress 8 results)
elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper"]].dropna()
    sub.columns = ["URN", "va", "lower", "upper"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * 1.96)
n_el = long.groupby("URN")["element"].nunique()
long = long[long["URN"].isin(n_el[n_el >= 3].index)].sort_values("URN").reset_index(drop=True)

# ---- all institutions: schools with GCSE results first, then institutions with A-level results only
urns_gcse = pd.Index(sorted(long["URN"].unique()))
urns_only = pd.Index(sorted(set(raw["URN"]) - set(urns_gcse)))
inst = urns_gcse.append(urns_only)
n_g, n_inst = len(urns_gcse), len(inst)
no_gcse = (np.arange(n_inst) >= n_g).astype(float)

long["school_idx"] = urns_gcse.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()
n_elements = len(elements)
x_obs, x_se = long["va"].to_numpy(), long["se"].to_numpy()
s_idx, e_idx = long["school_idx"].to_numpy(), long["element_idx"].to_numpy()

region_series = raw.drop_duplicates("URN").set_index("URN")["RGN24NM"].reindex(inst)
regions = list(region_series.value_counts().index)
reg_idx = region_series.map({r: k for k, r in enumerate(regions)}).fillna(len(regions)).astype(int).to_numpy()   # unknown region -> national average
is_london = (region_series.to_numpy() == "London")
print(f"{n_g} schools with GCSE results, {len(urns_only)} institutions with A-level results only, {n_inst} in all; {is_london.sum()} in London")


# ---- local authority for every institution
la_series = names["local_authority"].reindex(inst)
la_names = sorted(la_series.unique())
la_idx = la_series.map({a: k for k, a in enumerate(la_names)}).to_numpy()

# ---- institution type, from the public register's detailed establishment type
def type_category(t):
    if t in ("Other independent school", "Other independent special school"):
        return "independent (fee-paying)"
    if t in ("Academy converter", "Academy special converter"):
        return "academy converter"
    if t in ("Academy sponsor led", "Academy special sponsor led"):
        return "academy sponsor-led"
    if t in ("Free schools", "Free schools special", "University technical college", "Studio schools", "City technology college", "Free schools alternative provision"):
        return "free school / UTC / studio"
    if t in ("Community school", "Voluntary aided school", "Voluntary controlled school", "Foundation school"):
        return "LA maintained"
    if t in ("Further education", "Sixth form centres", "Academy 16-19 converter", "Academy 16 to 19 sponsor led", "Free schools 16 to 19"):
        return "college (post-16)"
    return "other"
type_names = ["independent (fee-paying)", "academy converter", "academy sponsor-led", "free school / UTC / studio", "LA maintained", "college (post-16)", "other"]
state4 = type_names[1:5]                      # state-funded types that have GCSE results
other3 = [type_names[0], type_names[5], type_names[6]]
inst_type = np.array([type_names.index(type_category(t)) for t in names["type_detail"].reindex(inst).to_numpy()])
is4 = ((inst_type >= 1) & (inst_type <= 4)).astype(float)
t4_idx = np.where(is4 == 1, inst_type - 1, 0)
is3 = 1.0 - is4
t3_idx = np.array([{0: 0, 5: 1, 6: 2}.get(k, 0) for k in inst_type])

# years as an academy (converters and sponsor-led), in decades, centred on a typical 12 years
open_year = pd.to_datetime(names["open_date"].reindex(inst), format="%d-%m-%Y", errors="coerce").dt.year.to_numpy()
is_cs = ((inst_type == 1) | (inst_type == 2)).astype(float)
cs_idx = np.where(inst_type == 2, 1, 0)      # 0 converter, 1 sponsor-led
years_c = is_cs * ((2024 - np.nan_to_num(open_year, nan=2012)) / 10.0 - 1.2)
summary = pd.crosstab(pd.Series(np.array(type_names)[inst_type], name="type"), pd.Series(np.where(no_gcse == 1, "A-level results only", "has GCSE results"), name=""))
print(summary.reindex(type_names).fillna(0).astype(int).to_string())

In [ ]:
arts = ["Art & Design", "Art & Design (Fine Art)", "Art & Design (Photography)", "Art & Design (Graphics)", "Art & Design (Textiles)",
        "Art & Design (3d Studies)", "Art & Design (Critical Studies)", "Music", "Music Technology", "Drama & Theatre Studies", "Dance"]
groups = {"Maths": ["Mathematics"],
          "Sciences": ["Biology", "Chemistry", "Physics"],
          "English": ["English Literature", "English Language", "English Language & Literature"],
          "Humanities": ["History", "Geography", "Religious Studies", "Logic/ Philosophy", "Ancient History", "Classical Civilisation"],
          "Social sciences": ["Psychology", "Sociology", "Economics", "Government & Politics", "Law"],
          "Business & Computing": ["Business Studies:Single", "Computer Studies/Computing"],
          "Creative arts": arts}
group_names = list(groups)
n_groups = len(group_names)

raw_i = raw.set_index("URN").reindex(inst)
frames = []
for gname, subjects in groups.items():
    va = pd.DataFrame({s: raw_i[f"A-level {s} VA"] for s in subjects})
    se = pd.DataFrame({s: (raw_i[f"A-level {s} VA upper"] - raw_i[f"A-level {s} VA lower"]) / (2 * 1.96) for s in subjects})
    ent = pd.DataFrame({s: raw_i[f"A-level {s} entries"].where(va[s].notna(), 0).fillna(0) for s in subjects})
    total = ent.sum(axis=1)
    pooled_va = (va.fillna(0) * ent).sum(axis=1) / total.replace(0, np.nan)
    pooled_se = np.sqrt(((se.fillna(0) * ent) ** 2).sum(axis=1)) / total.replace(0, np.nan)   # entry-weighted; assumes separate cohorts
    f = pd.DataFrame({"inst_idx": np.arange(n_inst), "group": gname, "va": pooled_va.to_numpy(), "se": pooled_se.to_numpy(), "entries": total.to_numpy()})
    frames.append(f.dropna(subset=["va", "se"]))
alevel = pd.concat(frames).reset_index(drop=True)
alevel["group_idx"] = alevel["group"].map({g: k for k, g in enumerate(group_names)}).to_numpy()
y_obs, y_se = alevel["va"].to_numpy(), alevel["se"].to_numpy()
ys_idx, yg_idx = alevel["inst_idx"].to_numpy(), alevel["group_idx"].to_numpy()
assert (y_se > 0).all()
print(f"{len(alevel)} institution-group A-level observations")

## Model

In [ ]:
keep = np.array([0.0 if e == "Humanities" else 1.0 for e in elements])

def build_model(state_types):
    """state_types=False: no type effects among the state types (only offsets for independent schools, colleges and others, as in the shortlist notebook).
       state_types=True: adds state-type shifts in GCSE quality, tilt, consistency and A-level value added, and time as an academy."""
    coords = {"element": elements, "region": regions, "group": group_names, "la": la_names, "type4": state4, "type3": other3,
              "cs": ["academy converter", "academy sponsor-led"]}
    with pm.Model(coords=coords) as model:
        # ---- geography (and, if wanted, state type): where general GCSE quality sits
        sigma_m = pm.HalfNormal("sigma_m", 0.5)
        m = pm.ZeroSumNormal("m", sigma=sigma_m, dims="region")
        m_all = pt.concatenate([m, pt.zeros(1)])
        sigma_a = pm.HalfNormal("sigma_a", 0.3)
        a_la = pm.Normal("a_la", 0, sigma_a, dims="la")
        g_loc = m_all[reg_idx] + a_la[la_idx]
        h_loc = np.zeros(n_inst)
        if state_types:
            type_g = pm.ZeroSumNormal("type_g", sigma=0.5, dims="type4")
            type_h = pm.ZeroSumNormal("type_h", sigma=0.5, dims="type4")
            type_c = pm.ZeroSumNormal("type_c", sigma=0.3, dims="type4")
            slope_g = pm.Normal("slope_g", 0, 0.3, dims="cs")
            g_loc = g_loc + is4 * type_g[t4_idx] + is_cs * years_c * slope_g[cs_idx]
            h_loc = is4 * type_h[t4_idx]

        # ---- GCSE side, schools with GCSE results only
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        g = pm.Normal("g", g_loc[:n_g], 1, shape=n_g)
        h = pm.Normal("h", h_loc[:n_g], 1, shape=n_g)
        k_raw = pm.Normal("kappa_raw", 0, 0.5, shape=n_elements)
        kappa = pm.Deterministic("kappa", k_raw * keep, dims="element")
        sigma_s = pm.HalfNormal("sigma_s", 0.5)
        rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
        w = pm.Normal("w", 0, 1, shape=n_g)
        log_s = sigma_s * (rho * (g - g_loc[:n_g]) + pt.sqrt(1 - rho**2) * w)
        if state_types:
            log_s = is4[:n_g] * type_c[t4_idx[:n_g]] + log_s
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx] + kappa[e_idx] * h[s_idx],
                  sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)
        g_full = pt.concatenate([g, g_loc[n_g:]])
        h_full = pt.concatenate([h, h_loc[n_g:]])

        # ---- A-level side (all institutions)
        nu = pm.Normal("nu", 0, 0.5, dims="group")
        sd_group = pm.HalfNormal("sd_group", 0.5, dims="group")
        sd_group_ng = pm.HalfNormal("sd_group_ng", 0.5, dims="group")
        sigma_psi = pm.HalfNormal("sigma_psi", 0.3)
        psi = pm.ZeroSumNormal("psi", sigma=sigma_psi, dims="region")
        psi_all = pt.concatenate([psi, pt.zeros(1)])
        sigma_xi = pm.HalfNormal("sigma_xi", 0.2)
        xi = pm.Normal("xi", 0, sigma_xi, dims="la")
        u = pm.Normal("u", 0, 1, shape=n_inst)
        b = pm.Normal("b", 0, 0.5, dims="group")
        c = pm.Normal("c", 0, 0.5, dims="group")
        lam_a = pm.HalfNormal("lam_a", 0.5, dims="group")
        d3 = pm.Normal("d3", 0, 0.5, dims=("group", "type3"))            # mean shift for independent schools, colleges and others, by group
        shared = psi_all[reg_idx] + xi[la_idx] + u
        y_mean = (nu[yg_idx] + b[yg_idx] * g_full[ys_idx] + c[yg_idx] * h_full[ys_idx] + lam_a[yg_idx] * shared[ys_idx]
                  + is3[ys_idx] * d3[yg_idx, t3_idx[ys_idx]])
        if state_types:
            slope_u = pm.Normal("slope_u", 0, 0.3, dims="cs")
            type_a = pm.ZeroSumNormal("type_a", sigma=0.3, dims=("group", "type4"))
            shared = shared + is_cs * years_c * slope_u[cs_idx]
            y_mean = (nu[yg_idx] + b[yg_idx] * g_full[ys_idx] + c[yg_idx] * h_full[ys_idx] + lam_a[yg_idx] * shared[ys_idx]
                      + is3[ys_idx] * d3[yg_idx, t3_idx[ys_idx]] + is4[ys_idx] * type_a[yg_idx, t4_idx[ys_idx]])
        sd_y = sd_group[yg_idx] * (1 - no_gcse[ys_idx]) + sd_group_ng[yg_idx] * no_gcse[ys_idx]
        pm.Normal("y_obs", mu=y_mean, sigma=pt.sqrt(sd_y**2 + y_se**2), observed=y_obs)
    return model

### Fitting and estimating

Each fit uses four chains of 500 draws after 1,500 tuning steps. After each fit the sign of the tilt is fixed (it is only defined up to sign), every institution's true value added in each subject group is estimated, and the raw fit is released before the next one, so the notebook fits in the machine's memory.

The true value added for institution $i$ in group $j$ is the model's expectation (GCSE profile where there is one, shared A-level quality, region and local authority, type) plus the group-specific departure, whose conditional distribution is the observed residual shrunk by $\sigma_j^2 / (\sigma_j^2 + \sigma_{ij}^2)$, so small cohorts are pulled hard and large cohorts barely at all.

In [ ]:
import gc
def align_and_extract(idata, state_types, thin=5):
    post = idata.posterior
    k = post["kappa"]
    score = (k.sel(element="Maths") + k.sel(element="Science") - k.sel(element="English") - k.sel(element="Open")).mean("draw")
    sign = xr.where(score > 0, 1.0, -1.0)
    for name in ["h", "kappa", "c"] + (["type_h"] if state_types else []):
        post[name] = post[name] * sign
    n_div = int(idata.sample_stats["diverging"].sum())
    ds = post.to_dataset()
    per_school = ["g", "h", "u", "xi", "a_la"]
    glob = [v for v in ds.data_vars if v not in per_school + ["w", "kappa_raw", "rho_raw"]]
    small = xr.Dataset({**{v: ds[v] for v in glob}, **{v: ds[v].isel({ds[v].dims[-1]: slice(0, None, 10)}) for v in ["g", "h", "u"]}})
    rh, es = az.rhat(small), az.ess(small)
    diag = pd.DataFrame({"max r_hat": {v: float(rh[v].max()) for v in rh.data_vars}, "min bulk ESS": {v: float(es[v].min()) for v in es.data_vars}}).sort_values("max r_hat", ascending=False)
    R = {}
    for v in glob + per_school:
        a = ds[v].to_numpy(); R[v] = a.reshape(-1, *a.shape[2:])[::thin]
    return R, diag, n_div, int((sign.to_numpy() < 0).sum())

def estimate_all(R, state_types, seed=1):
    """Posterior draws of the true value added of every institution-group observation; returns mean and 89% interval."""
    rg = np.random.default_rng(seed)
    n = len(R["nu"])
    m_all = np.concatenate([R["m"], np.zeros((n, 1))], axis=1)
    psi_all = np.concatenate([R["psi"], np.zeros((n, 1))], axis=1)
    g_loc = m_all[:, reg_idx] + R["a_la"][:, la_idx]
    shared_loc = psi_all[:, reg_idx] + R["xi"][:, la_idx]
    h_loc = np.zeros((n, n_inst))
    if state_types:
        g_loc = g_loc + is4[None, :] * R["type_g"][:, t4_idx] + (is_cs * years_c)[None, :] * R["slope_g"][:, cs_idx]
        h_loc = is4[None, :] * R["type_h"][:, t4_idx]
        shared_loc = shared_loc + (is_cs * years_c)[None, :] * R["slope_u"][:, cs_idx]
    g_full = np.concatenate([R["g"], g_loc[:, n_g:]], axis=1)
    h_full = np.concatenate([R["h"], h_loc[:, n_g:]], axis=1)
    idx, j = ys_idx, yg_idx
    mean = (R["nu"][:, j] + R["b"][:, j] * g_full[:, idx] + R["c"][:, j] * h_full[:, idx] + R["lam_a"][:, j] * (shared_loc[:, idx] + R["u"][:, idx])
            + is3[idx][None, :] * R["d3"][:, j, t3_idx[idx]])
    if state_types:
        mean = mean + is4[idx][None, :] * R["type_a"][:, j, t4_idx[idx]]
    sd2 = np.where(no_gcse[idx][None, :] == 1, R["sd_group_ng"][:, j], R["sd_group"][:, j]) ** 2
    k = sd2 / (sd2 + y_se[None, :] ** 2)
    true = mean + k * (y_obs[None, :] - mean) + np.sqrt(k) * y_se[None, :] * rg.standard_normal(mean.shape)
    return pd.DataFrame({"est": true.mean(axis=0), "lo": np.percentile(true, 5.5, axis=0), "hi": np.percentile(true, 94.5, axis=0)})

### Fit 1: without state types

In [ ]:
model_no = build_model(state_types=False)
with model_no:
    idata_no = pm.sample(draws=500, tune=1500, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)
R_no, diag_no, div_no, flip_no = align_and_extract(idata_no, False)
est_no = estimate_all(R_no, False)
del idata_no, model_no, R_no; gc.collect()
print(f"without state types: divergences = {div_no}, chains flipped = {flip_no} of 4")
display(diag_no.round(3).head(8))

### Fit 2: with state types

In [ ]:
model_yes = build_model(state_types=True)
with model_yes:
    idata_yes = pm.sample(draws=500, tune=1500, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)
R_yes, diag_yes, div_yes, flip_yes = align_and_extract(idata_yes, True)
est_yes = estimate_all(R_yes, True)
del idata_yes, model_yes, R_yes; gc.collect()
print(f"with state types: divergences = {div_yes}, chains flipped = {flip_yes} of 4")
display(diag_yes.round(3).head(8))

## How much do the estimates move?

One row per institution and subject group. `est_no` and `est` are the estimated true value added without and with state types, `shift` is the difference, and `published` is the raw published score with its standard error. The first table shows, for each subject group, how strongly the two sets of estimates agree and how large the shifts are; the second shows the average shift by type.

In [ ]:
res = alevel[["inst_idx", "group", "entries", "va", "se"]].rename(columns={"va": "published"}).reset_index(drop=True)
res["est_no"], res["lo_no"], res["hi_no"] = est_no["est"].to_numpy(), est_no["lo"].to_numpy(), est_no["hi"].to_numpy()
res["est"], res["lo"], res["hi"] = est_yes["est"].to_numpy(), est_yes["lo"].to_numpy(), est_yes["hi"].to_numpy()
res["shift"] = res["est"] - res["est_no"]
res["type"] = np.array(type_names)[inst_type[res["inst_idx"].to_numpy()]]
res["name"] = names.loc[inst[res["inst_idx"].to_numpy()], "name"].to_numpy()
res["region"] = region_series.to_numpy()[res["inst_idx"].to_numpy()]
res["URN"] = inst[res["inst_idx"].to_numpy()]

def top_overlap(d, k=20):
    a, b = set(d.nlargest(k, "est_no")["inst_idx"]), set(d.nlargest(k, "est")["inst_idx"])
    return len(a & b)
rows = []
for gname in group_names:
    d = res[res["group"] == gname]
    dl = d[d["region"] == "London"]
    rows.append({"group": gname, "institutions": len(d), "correlation": np.corrcoef(d["est_no"], d["est"])[0, 1], "mean |shift|": d["shift"].abs().mean(),
                 "share moving > 0.05": (d["shift"].abs() > 0.05).mean(), "largest up": d["shift"].max(), "largest down": d["shift"].min(),
                 "top 20 kept, England": top_overlap(d), "top 20 kept, London": top_overlap(dl)})
display(pd.DataFrame(rows).set_index("group").round(3))
by_type = res.pivot_table(index="type", columns="group", values="shift", aggfunc="mean").reindex(type_names)[group_names]
counts = res.groupby("type")["inst_idx"].nunique().reindex(type_names)
by_type["institutions"] = counts
display(by_type.round(3))

## The scatter, with and without type

For each subject group, every institution as a point: its **estimate without state types** (x) against its **estimate with them** (y). Points on the diagonal have not moved. Colour is the type of institution and marker size the cohort. The three biggest movers up and the three biggest movers down in each panel are labelled. The first figure is all of England, the second is London only.

In [ ]:
type_colour = {"independent (fee-paying)": "#DD8452", "academy converter": "#4C72B0", "academy sponsor-led": "#C44E52", "free school / UTC / studio": "#55A868",
               "LA maintained": "#8172B3", "college (post-16)": "#937860", "other": "#999999"}
def short(n, k=26): return n if len(n) <= k else n[:k - 1].rstrip(" ,") + "…"

def move_scatter(mask, title, filename):
    fig, axes = plt.subplots(2, 4, figsize=(23, 11)); axes = axes.ravel()
    for ax, gname in zip(axes, group_names):
        d = res[(res["group"] == gname) & mask[res["inst_idx"].to_numpy()]]
        for t in type_names:
            s = d[d["type"] == t]
            if len(s):
                ax.scatter(s["est_no"], s["est"], s=8 + 5 * np.sqrt(s["entries"]), color=type_colour[t], alpha=0.5, edgecolor="white", linewidth=0.3, label=t)
        lo_, hi_ = d[["est_no", "est"]].min().min() - 0.05, d[["est_no", "est"]].max().max() + 0.05
        ax.plot([lo_, hi_], [lo_, hi_], color="grey", linewidth=0.9, linestyle="--")
        ax.set_xlim(lo_, hi_); ax.set_ylim(lo_, hi_)
        movers = pd.concat([d.nlargest(3, "shift"), d.nsmallest(3, "shift")])
        for _, r in movers.iterrows():
            ax.annotate(f"{short(r['name'])} ({int(r['entries'])})", (r["est_no"], r["est"]), xytext=(6, -3), textcoords="offset points", fontsize=7)
        ax.set_title(f"{gname}  (n = {len(d)})", loc="left"); ax.set_xlabel("estimate without state types"); ax.set_ylabel("estimate with state types")
    handles, labels = axes[0].get_legend_handles_labels()
    axes[7].axis("off"); axes[7].legend(handles, labels, loc="center", fontsize=11, title="type; marker size = cohort", title_fontsize=11, frameon=False)
    fig.suptitle(title, fontsize=14, x=0.01, ha="left")
    plt.tight_layout()
    plt.savefig(filename, dpi=140, bbox_inches="tight")
    plt.show()

import os
os.makedirs("results", exist_ok=True)
move_scatter(np.ones(n_inst, dtype=bool), "Estimated value added: without state types (x) and with them (y), England", "results/estimate-shift-england.png")

In [ ]:
move_scatter(is_london, "Estimated value added: without state types (x) and with them (y), London", "results/estimate-shift-london.png")

## The familiar scatter: published against estimated, with type

The published-against-estimated scatter of the shortlist notebook, now with the type-aware estimate on the y axis and coloured by type. The grey points behind are the estimates *without* state types, so the gap between a grey and a coloured point at the same published value is the movement, seen from the published axis. Independent schools (orange) and sponsor-led academies (red) are the types most likely to have moved.

In [ ]:
def published_scatter(mask, title, filename):
    fig, axes = plt.subplots(2, 4, figsize=(23, 11)); axes = axes.ravel()
    for ax, gname in zip(axes, group_names):
        d = res[(res["group"] == gname) & mask[res["inst_idx"].to_numpy()]]
        ax.scatter(d["published"], d["est_no"], s=6, color="#BBBBBB", alpha=0.5, label="estimate without state types", zorder=1)
        for t in type_names:
            s = d[d["type"] == t]
            if len(s):
                ax.scatter(s["published"], s["est"], s=8 + 5 * np.sqrt(s["entries"]), color=type_colour[t], alpha=0.55, edgecolor="white", linewidth=0.3, label=t, zorder=2)
        lo_, hi_ = d["published"].min() - 0.05, d["published"].max() + 0.05
        ax.plot([lo_, hi_], [lo_, hi_], color="grey", linewidth=0.9, linestyle="--")
        ax.axhline(0, color="lightgrey", linewidth=0.7); ax.axvline(0, color="lightgrey", linewidth=0.7)
        ax.set_title(f"{gname}  (n = {len(d)})", loc="left"); ax.set_xlabel("published value added"); ax.set_ylabel("estimated true value added")
    handles, labels = axes[0].get_legend_handles_labels()
    axes[7].axis("off"); axes[7].legend(handles, labels, loc="center", fontsize=11, title="colour = type, with state types", title_fontsize=11, frameon=False)
    fig.suptitle(title, fontsize=14, x=0.01, ha="left")
    plt.tight_layout()
    plt.savefig(filename, dpi=140, bbox_inches="tight")
    plt.show()
published_scatter(np.ones(n_inst, dtype=bool), "Published against estimated value added, England (grey: without state types)", "results/published-vs-estimate-with-type-england.png")

In [ ]:
published_scatter(is_london, "Published against estimated value added, London (grey: without state types)", "results/published-vs-estimate-with-type-london.png")

## Who moves, by type

The distribution of the shift (with state types minus without) for each type, by subject group. Boxes show the middle half of institutions, whiskers the 5th to 95th percentile.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(23, 9)); axes = axes.ravel()
plot_types = [t for t in type_names]
for ax, gname in zip(axes, group_names):
    d = res[res["group"] == gname]
    data = [d.loc[d["type"] == t, "shift"].to_numpy() for t in plot_types]
    bp = ax.boxplot([x if len(x) else [np.nan] for x in data], orientation="horizontal", whis=(5, 95), showfliers=False, patch_artist=True, tick_labels=[t[:20] for t in plot_types])
    for patch, t in zip(bp["boxes"], plot_types): patch.set_facecolor(type_colour[t]); patch.set_alpha(0.7)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--"); ax.invert_yaxis()
    ax.set_title(gname, loc="left"); ax.set_xlabel("shift in estimate (points)")
axes[7].axis("off")
plt.tight_layout(); plt.show()

## Summary

**The estimates barely move.** Including the state types leaves every institution's estimated value added almost where it was:

- The two sets of estimates correlate at 0.999 or above in every subject group, the average absolute shift is 0.004 to 0.010 points, and only 0.1% to 2.3% of institutions move by more than 0.05 (Business & Computing 2.3%, Maths 1.4%, Sciences 0.7%, the rest 0.2% or less).
- The largest single shifts are +0.025 (Creative arts) to +0.063 (Maths) and -0.040 (English) to -0.101 (Maths).
- **The top of each list does not change:** 19 or 20 of the top 20 institutions in England are the same in every group, and in London 18 to 20 (18 in Maths, 19 in Sciences, English, Humanities and Business & Computing, 20 in Social sciences and Creative arts).
- In the published-against-estimated plots the grey "without state types" points are completely hidden behind the coloured "with state types" ones: the y axis has not moved.

**The movement that exists is systematic by type, and small.** Average shift by type (points):

| | Maths | Sciences | English | Humanities | Social sciences | Business & Computing | Creative arts |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Academy sponsor-led | -0.008 | -0.015 | -0.008 | -0.018 | +0.003 | -0.041 | -0.025 |
| Free school / UTC / studio | -0.037 | -0.028 | +0.026 | +0.007 | +0.018 | -0.001 | +0.006 |
| Academy converter | +0.001 | +0.008 | +0.002 | +0.001 | -0.003 | +0.004 | +0.003 |
| LA maintained | +0.014 | -0.016 | -0.005 | +0.002 | +0.004 | +0.008 | +0.001 |
| Independent | +0.001 | 0.000 | -0.001 | 0.000 | 0.000 | -0.001 | -0.001 |

Sponsor-led academies move down a little in most groups, most in Business & Computing (-0.04) and Creative arts (-0.025), and converters move up slightly. Free schools, UTCs and studio schools move in different directions by subject (down in Maths and Sciences, up in English and Social sciences), which fits their different specialisms. Independent schools and colleges do not move, because they had their own offsets in both fits. The biggest individual movers are small cohorts of 6 to 11 entries, mostly free schools, UTCs and studio schools and sponsor-led academies (in Maths, for example, The Swan School, William de Ferrers School, Barnwell School and Pioneer Secondary Academy), because the model leans on its expectation more when the published score is thin.

**Why it moves so little.** State type changes what the model expects of an institution by a few hundredths of a point up to about 0.1 at most, but an institution's own published result carries about half the weight in its estimate for a typical cohort (for Maths, roughly 0.45), and it is the results that the estimate follows. The model without state types also already carried much of the information type would add, through the institution's own GCSE profile, its region and local authority, and its shared A-level quality.

**What this means.** Type matters for reading the results: it explains why sponsor-led academies score lower on average and what part of the converter versus sponsor-led gap is GCSE and what part is beyond it (`a-level-institution-type.ipynb`). But it does **not** change the shortlists. An institution's place in a list follows its own published results and cohort size, and the shortlists in `a-level-england-shortlist.ipynb` would look the same with type included: at most one or two names in a top 20 change.

**Sampling.** Without state types: 1 divergence in 2,000 draws, worst $\hat R$ 1.06 (the overall GCSE level $\mu_e$, ESS 45). With state types: no divergences, worst $\hat R$ 1.11 ($\mu_e$, ESS 27), with the regional shifts and their spread weakest after that (ESS 35 to 45) and the rest above 60. These are the parameters that trade off against each other; the estimates of institutions' value added depend on combinations that are better determined, and are the same in the two runs of this notebook.

**Caveats.** This is for the seven subject groups (Economics and Psychology sit inside Social sciences), on estimates from one fit of each model. Shifts here are in value-added points; the second fit's type effects are as in the type notebook.